In [1]:
import numpy as np
import pandas as pd

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [5]:
df = pd.read_csv('covid_toy.csv')

In [7]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [13]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [17]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2)

In [29]:
X_train

,age,gender,fever,cough,city
9,64,Female,101.0,Mild,Delhi
97,20,Female,101.0,Mild,Bangalore
21,73,Male,98.0,Mild,Bangalore
41,82,Male,NaN,Mild,Kolkata
56,71,Male,NaN,Strong,Kolkata
...,...,...,...,...,...
74,34,Female,104.0,Strong,Delhi
1,27,Male,100.0,Mild,Delhi
81,65,Male,99.0,Mild,Delhi
79,48,Female,103.0,Mild,Kolkata


## 1. Normal Way- without using column transformer

In [46]:
# adding simple imputer to fever col
si = SimpleImputer() # used to fill missing values
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.transform(X_test[['fever']])

In [56]:
X_train_fever.shape

(80, 1)

In [42]:
X_train['fever']

9     101.0
97    101.0
21     98.0
41      NaN
56      NaN
      ...  
74    104.0
1     100.0
81     99.0
79    103.0
58     98.0
Name: fever, Length: 80, dtype: float64

In [44]:
X_train[['fever']]

,fever
9,101.0
97,101.0
21,98.0
41,NaN
56,NaN
...,...
74,104.0
1,100.0
81,99.0
79,103.0


In [69]:
# Ordinal encoding --> cough
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [79]:
# OneHotEncoding --> gender, city
ohe = OneHotEncoder(drop='first', sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])

# also the test data
X_test_gender_city = ohe.transform(X_test[['gender', 'city']])

X_train_gender_city.shape

(80, 4)

In [93]:
# Extracting age
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values

X_train_age.shape

(80, 1)

In [95]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough), axis=1)

# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough), axis=1)

X_train_transformed.shape

(80, 7)

In [99]:
X_train_transformed

array([[ 64.        , 101.        ,   0.        ,   1.        ,
          0.        ,   0.        ,   0.        ],
       [ 20.        , 101.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ],
       [ 73.        ,  98.        ,   1.        ,   0.        ,
          0.        ,   0.        ,   0.        ],
       [ 82.        , 100.84507042,   1.        ,   0.        ,
          1.        ,   0.        ,   0.        ],
       [ 71.        , 100.84507042,   1.        ,   0.        ,
          1.        ,   0.        ,   1.        ],
       [ 74.        , 102.        ,   1.        ,   0.        ,
          0.        ,   1.        ,   0.        ],
       [ 42.        , 101.        ,   1.        ,   1.        ,
          0.        ,   0.        ,   0.        ],
       [ 69.        ,  98.        ,   0.        ,   0.        ,
          0.        ,   1.        ,   1.        ],
       [ 69.        , 103.        ,   0.        ,   0.        ,
          1.    

## 2. Best way

In [102]:
from sklearn.compose import ColumnTransformer

In [110]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(),['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False,drop='first'), ['gender', 'city'])
],remainder='passthrough')

In [112]:
transformer.fit_transform(X_train)

array([[101.        ,   0.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  64.        ],
       [101.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  20.        ],
       [ 98.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  73.        ],
       [100.84507042,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  82.        ],
       [100.84507042,   1.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  71.        ],
       [102.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   1.        ,  74.        ],
       [101.        ,   0.        ,   1.        ,   1.        ,
          0.        ,   0.        ,  42.        ],
       [ 98.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  69.        ],
       [103.        ,   0.        ,   0.        ,   0.        ,
          1.    

In [114]:
transformer.fit_transform(X_train).shape

(80, 7)

In [116]:
transformer.transform(X_test)

array([[ 98.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  81.        ],
       [ 98.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  24.        ],
       [ 98.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  64.        ],
       [103.        ,   1.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  46.        ],
       [ 99.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  49.        ],
       [103.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  83.        ],
       [104.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  75.        ],
       [103.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  60.        ],
       [100.        ,   1.        ,   0.        ,   0.        ,
          1.    

In [118]:
transformer.transform(X_test).shape

(20, 7)